# EHRSQL-2024 (Gemma 4) — fine-tune + evaluate on Colab → adapter for local RTX 3090

**Benchmark:** EHRSQL-2024 (ClinicalNLP@NAACL 2024), **MIMIC-IV Demo**, scored with the
**official Reliability Score** — apples-to-apples with the leaderboard **RS(10) = 0.8132**.
**Policy:** Colab does fine-tuning + evaluation only; final inference runs on the local RTX 3090.

**Pipeline:** prepare augmented SFT (≈44k) → QLoRA SFT (Gemma 4 12B) → Abstention-ORPO pairs →
Abstention-ORPO → eval on the 1,167-question test set with the official RS scorer.

**Model menu** (largest that fits a 24 GB 3090 for inference at 4-bit = **12B**):

| model | HF id | 4-bit (inference) |
|---|---|---|
| Gemma 4 12B (recommended, max for 24GB) | `unsloth/gemma-4-12b-it` | 6.7 GB |
| Gemma 4 E4B (faster) | `unsloth/gemma-4-E4B-it` | 4.5 GB |

**Drive flow:** code+data come zipped from `MyDrive/ehrsql/EHRSQL_GEMMA/ehrsql_gemma_bundle.zip`
→ unzipped to local Colab disk. Adapters + RS results written **back** there. Add a Colab
**Secret `HF_TOKEN`** first.

## 1. Mount Drive

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

## 2. Copy + unzip the bundle from the shared folder to local disk

In [ ]:
import os
# The 'ehrsql' folder in the Colab account's Drive holds a shortcut to the shared
# folder 'EHRSQL_GEMMA' (owned by your 5TB account).
DRIVE_DIR = '/content/drive/MyDrive/ehrsql/EHRSQL_GEMMA'
BUNDLE = f'{DRIVE_DIR}/ehrsql_gemma_bundle.zip'
assert os.path.exists(BUNDLE), f'Not found: {BUNDLE} (upload the zip + share the folder first)'
os.makedirs('/content/ehrcopilot', exist_ok=True)
!unzip -q -o "$BUNDLE" -d /content/ehrcopilot
os.chdir('/content/ehrcopilot')
print('cwd:', os.getcwd()); !ls

## 3. Install dependencies (Gemma 4 = transformers 5.10.1 + Unsloth-from-git)

In [ ]:
# Gemma 4 = model_type `gemma4_unified`, which needs transformers>=5.10. The current
# git unsloth_zoo pins transformers<=5.5.0 (too conservative), so:
#   1) install the Unsloth stack (this pulls transformers 5.5.0),
#   2) FORCE transformers==5.10.1 with --no-deps to override that cap.
# Unsloth's official Gemma4_(12B)_Text notebook validated exactly transformers==5.10.1
# with these git packages, so the override is safe (the pip 'requires <=5.5' warning is expected).
!pip -q install "unsloth @ git+https://github.com/unslothai/unsloth" \
               "unsloth_zoo @ git+https://github.com/unslothai/unsloth-zoo" \
               bitsandbytes accelerate datasets sentencepiece timm \
               scikit-learn rank-bm25 sqlglot faiss-cpu einops
!pip -q install --no-deps --force-reinstall "transformers==5.10.1"
import transformers; print('transformers', transformers.__version__, '(need >=5.10 for gemma4)')

## 4. HF token (Colab Secret) + environment + pick model + scale batch to GPU

In [ ]:
import os
from google.colab import userdata
os.environ['HF_TOKEN'] = userdata.get('HF_TOKEN')          # gated Gemma 4 access
os.environ['PYTHONPATH'] = 'src'
os.environ['PYTORCH_CUDA_ALLOC_CONF'] = 'expandable_segments:True'
os.environ['TOKENIZERS_PARALLELISM'] = 'false'

# >>> choose the model (12B = max that runs inference on the 24GB 3090) <<<
MODEL = 'unsloth/gemma-4-12b-it'   # TEXT model, FastModel path, max that fits a 24GB 3090 at 4-bit
SFT_EPOCHS = 1     # augmented train ≈44k (8x the raw 5k) → 1 epoch ≈ the old 3 epochs of updates; bump to 2 for more
ORPO_EPOCHS = 2
MAX_ANSWERABLE = 1500   # answerable ORPO pairs (each = 1 model sample); unanswerable pairs are free

import torch
p = torch.cuda.get_device_properties(0); VRAM = p.total_memory/1e9
print(f'GPU: {p.name}  {VRAM:.0f} GB   MODEL={MODEL}')
BS, GA = (16,1) if VRAM>70 else (8,2) if VRAM>38 else (2,8) if VRAM>20 else (1,16)
print('batch_size', BS, 'grad_accum', GA, '(effective', BS*GA, ')')

## 5. Prepare augmented SFT data (regenerate the ~137 MB JSONL from the shipped train_aug)

In [ ]:
import os
D = 'data/ehrsql2024/mimic_iv'
assert os.path.exists(f'{D}/mimic_iv.sqlite'), 'missing 2024 sqlite — rebuild the bundle'
# train_aug/{data.json,label.json} ships in the bundle; the big SFT JSONL is regenerated here (~30 s).
if not os.path.exists('data/ehrsql2024/sft_train_aug.jsonl'):
    !PYTHONPATH=src python -m ehrcopilot.finetune.prepare_sft \
      --train {D}/train_aug --valid {D}/valid --output data/ehrsql2024/sft_train_aug.jsonl
!wc -l data/ehrsql2024/sft_train_aug.jsonl

## 6. QLoRA SFT (Gemma 4) on augmented EHRSQL-2024

In [ ]:
!PYTHONPATH=src python -m ehrcopilot.finetune.qlora_sft \
  --base-model {MODEL} --data data/ehrsql2024/sft_train_aug.jsonl --output checkpoints/sft_g4_2024 \
  --epochs {SFT_EPOCHS} --batch-size {BS} --grad-accum {GA}

## 7. Build Abstention-ORPO pairs (unanswerable→[ABSTAIN], answerable→exec-verified)

In [ ]:
!PYTHONPATH=src python -m ehrcopilot.finetune.build_pairs \
  --train data/ehrsql2024/mimic_iv/train_aug --valid data/ehrsql2024/mimic_iv/valid \
  --adapter checkpoints/sft_g4_2024/adapter_final --output data/ehrsql2024/orpo_pairs.jsonl \
  --max-answerable {MAX_ANSWERABLE} --verify-execution

## 8. Abstention-ORPO (calibrates [ABSTAIN] → positive RS)

In [ ]:
!PYTHONPATH=src python -m ehrcopilot.finetune.abstention_dpo \
  --pairs data/ehrsql2024/orpo_pairs.jsonl --adapter checkpoints/sft_g4_2024/adapter_final \
  --output checkpoints/orpo_g4_2024 --epochs {ORPO_EPOCHS} --max-length 1536

## 9. GPU step (Colab): generate predictions + confidence FEATURES for valid + test
The abstention-gate calibration runs OFFLINE on CPU locally (`scripts/eda/calibrate_gate.py`); Colab only does the GPU generation. ~3 h/split on the big GPU.

In [ ]:
# Emit predictions + confidence features (max-entropy, bottom-k log-prob) for BOTH splits.
# exec-gate=error abstains on un-executable SQL (the free RS(10) lever); the confidence gate is
# calibrated locally afterwards from the dumped features.
ADAPTER = 'checkpoints/sft_g4_2024/adapter_final'   # primary gate-calibration target (also try orpo)
for split in ['valid', 'test']:
    !PYTHONPATH=src python -m ehrcopilot.eval.eval2024 data/ehrsql2024/mimic_iv/{split} \
      --model {ADAPTER} --few-shot data/ehrsql2024/mimic_iv/train \
      --retrieval-mode embed --repair --exec-gate error --dump-features \
      --output tests/evalgen/g4_2024_sft_{split}.json
import json
for split in ['valid', 'test']:
    d = json.load(open(f'tests/evalgen/g4_2024_sft_{split}.json'))
    print(f"{split:5s}  EX={d['EX']}  RS(0)={d['RS(0)']}  RS(10)={d['RS(10)']}  (exec-gate only; conf-gate added locally)")

## 10. Save adapters + predictions + FEATURES back to Drive (calibrate the gate locally)

In [ ]:
DEST = f'{DRIVE_DIR}/checkpoints'
!mkdir -p "$DEST" "$DRIVE_DIR"/results
!cp -r checkpoints/sft_g4_2024  "$DEST"/ 2>/dev/null || true
!cp -r checkpoints/orpo_g4_2024 "$DEST"/ 2>/dev/null || true
# metrics + prediction.json + features.json for valid & test -> needed for local gate calibration
!cp tests/evalgen/g4_2024_sft_*.json "$DRIVE_DIR"/results/ 2>/dev/null || true
print('Saved. Download results/g4_2024_sft_{valid,test}.{prediction,features}.json, then locally run:')
print('  .venv/bin/python scripts/eda/calibrate_gate.py \\')
print('    --valid-pred tests/evalgen/g4_2024_sft_valid.prediction.json --valid-feat tests/evalgen/g4_2024_sft_valid.features.json \\')
print('    --test-pred  tests/evalgen/g4_2024_sft_test.prediction.json  --test-feat  tests/evalgen/g4_2024_sft_test.features.json')

## Local inference (RTX 3090)

Download `EHRSQL_GEMMA/checkpoints/orpo_g4_2024/adapter_final` into `checkpoints/orpo_g4_2024/`,
then evaluate/serve locally (4-bit, ~6.7 GB):

```bash
PYTHONPATH=src python -m ehrcopilot.eval.eval2024 data/ehrsql2024/mimic_iv/test \
  --model checkpoints/orpo_g4_2024/adapter_final --few-shot data/ehrsql2024/mimic_iv/train \
  --retrieval-mode embed --repair --output tests/evalgen/g4_2024_orpo_local.json
```